<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/advanced/01_distributed_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Distributed Training & Large-Scale ML

Train models across multiple GPUs and machines with PyTorch DDP, mixed precision, gradient accumulation, and ZeRO optimization.

**Topics:** PyTorch DDP, mixed precision (AMP), gradient accumulation, ZeRO stages, model parallelism, training profiling

## 1. PyTorch Distributed Data Parallel (DDP) Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler

def setup_ddp(rank: int, world_size: int, backend: str = 'nccl'):
    """Initialize the distributed process group.
    
    nccl: NVIDIA Collective Communications Library — fastest for GPU-GPU
    gloo: CPU-based, works without CUDA — good for debugging
    """
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group(backend, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp():
    dist.destroy_process_group()

class MLModel(nn.Module):
    def __init__(self, input_dim: int = 512, hidden: int = 2048, output_dim: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, output_dim),
        )
    def forward(self, x): return self.net(x)

def train_one_epoch_ddp(rank: int, world_size: int):
    """Full DDP training loop for one epoch."""
    setup_ddp(rank, world_size)
    
    model = MLModel().cuda(rank)
    model = DDP(model, device_ids=[rank])  # wrap in DDP
    
    dataset = TensorDataset(torch.randn(1000, 512), torch.randint(0, 10, (1000,)))
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    loader = DataLoader(dataset, batch_size=32, sampler=sampler)  # each GPU sees 1/N data
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    for X, y in loader:
        X, y = X.cuda(rank), y.cuda(rank)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()   # DDP automatically syncs gradients across GPUs
        optimizer.step()
    
    cleanup_ddp()

# Launch: torch.multiprocessing.spawn(train_one_epoch_ddp, args=(4,), nprocs=4)
print('DDP key insight: each GPU trains on its own data shard.')
print('After backward(), DDP all-reduces gradients across all GPUs.')
print('Effective batch size = per_gpu_batch_size × world_size')
print()
n_gpus = torch.cuda.device_count()
print(f'Available GPUs: {n_gpus} (need >=2 for DDP)')

## 2. Mixed Precision Training with torch.cuda.amp

In [ ]:
from torch.cuda.amp import autocast, GradScaler
import time

def train_with_amp(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    epochs: int = 5,
):
    """Mixed precision training — 2x speedup on Volta/Ampere GPUs.
    
    FP16 computation (fast) + FP32 master weights (stable) = best of both worlds.
    GradScaler prevents FP16 underflow (tiny gradients becoming 0).
    """
    scaler = GradScaler()  # scales loss to prevent FP16 underflow
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        for X, y in loader:
            optimizer.zero_grad()
            
            with autocast():  # forward pass in FP16
                outputs = model(X)
                loss = criterion(outputs, y)
            
            scaler.scale(loss).backward()   # backward in FP16 with scaled loss
            
            # Unscale before gradient clipping (important!)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)   # skips step if gradients are inf/nan
            scaler.update()          # adjusts scale factor for next iteration

# Memory and speed benefits
comparison = {
    'FP32 model (1B params)': '4 GB VRAM',
    'FP16 model (1B params)': '2 GB VRAM',
    'BF16 model (1B params)': '2 GB VRAM (more numerically stable)',
    'INT8 quantized':         '1 GB VRAM (inference only)',
    'INT4 quantized':         '0.5 GB VRAM (inference only, some accuracy loss)',
}
print('Memory usage for 1B parameter model:')
for dtype, mem in comparison.items():
    print(f'  {dtype:<30}: {mem}')
print()
print('Typical speedup with AMP: 1.5-3x on modern NVIDIA GPUs')
print('Use BF16 (not FP16) on A100/H100 — wider dynamic range, no scaler needed.')

## 3. Gradient Accumulation for Large Effective Batch Sizes

In [ ]:
def train_with_gradient_accumulation(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    accumulation_steps: int = 8,  # effective batch = micro_batch × 8
):
    """Simulate large batch training on small GPU.
    
    Use case: You want batch_size=256 but only fit 32 per GPU.
    Solution: Accumulate gradients over 8 micro-batches before stepping.
    """
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()
    
    optimizer.zero_grad()
    
    for step, (X, y) in enumerate(loader):
        with autocast():
            loss = criterion(model(X), y)
            loss = loss / accumulation_steps  # normalize: same scale as full batch
        
        scaler.scale(loss).backward()  # accumulate gradients
        
        if (step + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)  # update once per accumulation_steps
            scaler.update()
            optimizer.zero_grad()

# Example: training a 7B LLM on a single 24GB GPU
scenarios = [
    ('7B LLM, single A100 80GB', 8,   8,   64,  'Feasible'),
    ('7B LLM, single RTX 4090', 2,   16,  32,  'Tight — use QLoRA'),
    ('1B model, single V100',   32,  4,   128, 'Comfortable'),
]
print(f'{"Setup":<30} {"Micro BS":<10} {"Accum":<8} {"Eff BS":<8} {"Status"}')
print('-' * 65)
for setup, micro, accum, eff, status in scenarios:
    print(f'{setup:<30} {micro:<10} {accum:<8} {eff:<8} {status}')
print()
print('Rule: loss = loss / accumulation_steps prevents gradient magnitude mismatch.')

## 4. ZeRO Optimizer — DeepSpeed Configuration

In [ ]:
import json

# DeepSpeed ZeRO stages — memory reduction strategies
ZERO_CONFIGS = {
    'zero_stage_1': {
        'zero_optimization': {
            'stage': 1,  # partition optimizer state across GPUs
            # Reduces optimizer memory by world_size
        },
        'fp16': {'enabled': True},
        'train_batch_size': 32,
    },
    'zero_stage_2': {
        'zero_optimization': {
            'stage': 2,  # partition optimizer state + gradients
            'allgather_partitions': True,
            'overlap_comm': True,  # overlap gradient comm with backward pass
        },
        'bf16': {'enabled': True},
        'train_batch_size': 32,
    },
    'zero_stage_3': {
        'zero_optimization': {
            'stage': 3,  # partition optimizer state + gradients + model params
            'offload_optimizer': {'device': 'cpu'},  # offload to RAM!
            'offload_param': {'device': 'cpu'},
        },
        'bf16': {'enabled': True},
        'train_batch_size': 64,
    },
}

# Memory savings per GPU (8-GPU setup, 7B model)
zero_memory_savings = [
    ('No ZeRO (DDP)',  '~112 GB', 'Full replica on each GPU'),
    ('ZeRO Stage 1',  '~56 GB',  'Optimizer states sharded (8x reduction)'),
    ('ZeRO Stage 2',  '~28 GB',  '+ Gradients sharded'),
    ('ZeRO Stage 3',  '~14 GB',  '+ Parameters sharded'),
    ('ZeRO Stage 3+', '~7 GB',   '+ CPU offloading'),
]
print('ZeRO memory savings for 7B model on 8 GPUs:')
print(f'{"Strategy":<20} {"Per-GPU Memory":<16} {"Notes"}')
print('-' * 60)
for strategy, mem, notes in zero_memory_savings:
    print(f'{strategy:<20} {mem:<16} {notes}')
print()
print('Launch: deepspeed --num_gpus=8 train.py --deepspeed ds_config.json')

## 5. Profiling and Benchmarking Distributed Training

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity
import time
from dataclasses import dataclass

@dataclass
class TrainingBenchmark:
    strategy: str
    throughput_samples_per_sec: float
    gpu_memory_gb: float
    time_to_converge_hrs: float

def benchmark_training_step(
    model: nn.Module,
    batch: tuple,
    n_warmup: int = 5,
    n_measure: int = 20,
) -> dict:
    """Measure throughput, latency, and memory usage of a training step."""
    X, y = batch
    optimizer = torch.optim.AdamW(model.parameters())
    criterion = nn.CrossEntropyLoss()
    
    # Warmup (fill GPU pipeline)
    for _ in range(n_warmup):
        optimizer.zero_grad()
        criterion(model(X), y).backward()
        optimizer.step()
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_measure):
        optimizer.zero_grad()
        criterion(model(X), y).backward()
        optimizer.step()
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    elapsed = time.perf_counter() - start
    samples_per_sec = (n_measure * X.shape[0]) / elapsed
    
    return {
        'ms_per_step': elapsed / n_measure * 1000,
        'samples_per_sec': samples_per_sec,
        'gpu_memory_gb': torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0,
    }

# Simulated benchmark results
benchmarks = [
    TrainingBenchmark('Single GPU (FP32)',  320,  16.0, 48.0),
    TrainingBenchmark('Single GPU (AMP)',   620,  9.5,  25.0),
    TrainingBenchmark('4-GPU DDP (AMP)',    2200, 9.5,  7.0),
    TrainingBenchmark('8-GPU DDP (AMP)',    4100, 9.5,  3.5),
    TrainingBenchmark('8-GPU + ZeRO-2',    3800, 5.2,  3.8),
]

print(f'{"Strategy":<25} {"Samples/sec":<14} {"GPU Mem (GB)":<14} {"Time (hrs)"}')
print('-' * 68)
for b in benchmarks:
    print(f'{b.strategy:<25} {b.throughput_samples_per_sec:<14.0f} {b.gpu_memory_gb:<14.1f} {b.time_to_converge_hrs:.1f}')

## Exercises

1. **DDP from scratch:** Implement a simple parameter server (without PyTorch DDP) where one process aggregates gradients from workers using `dist.reduce()`. Compare convergence with DDP's `all_reduce()`.

2. **AMP investigation:** Train a small model with FP32, FP16, and BF16. Plot training loss curves and measure wall-clock time per epoch. Identify which dtype causes instability and why.

3. **Memory profiler:** Use `torch.profiler` to generate a Chrome trace of a training step. Identify the top-3 memory-consuming operations and suggest optimizations.